In [ ]:
import sys, os, io, glob, contextlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Add project root to path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from edge import compute_edge
from critic_model import (
    build_critic_profiles,
    build_kde_lambda_model,
    default_training_slugs,
    estimate_lambda,
    estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"
THRESHOLDS = list(range(45, 100, 5))

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# --- Load data ---

reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)

movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

print(f"Reviews: {len(reviews_df):,} rows, {reviews_df['movie_slug'].nunique()} movies")
print(f"Movies index: {len(movies_df)} movies")

# Slugs with hourly price data
slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])
movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")
print(f"Movies with price data: {len(movies_bt)}")

## Helper functions + backtest

Identical to `kde_backtest.ipynb`. Runs daily snapshots (~2-5 min).

In [ ]:
def load_hourly_prices(slug):
    """Load and forward-fill the hourly price CSV for a movie."""
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files:
        return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    df[thresh_cols] = df[thresh_cols].ffill()
    return df


def get_resolution(price_df):
    """Derive resolution from terminal prices. Returns {threshold: bool or None}."""
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    resolution = {}
    for col in thresh_cols:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty:
            resolution[thresh] = None
            continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90:
            resolution[thresh] = True
        elif terminal <= 10:
            resolution[thresh] = False
        else:
            resolution[thresh] = None
    return resolution


def precompute_review_states(slug, reviews_df, bet_close):
    """Precompute cumulative review states sorted by timestamp."""
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    
    if movie_reviews.empty:
        return [], None
    
    states = []
    critics = set()
    fresh = 0
    total = 0
    
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive":
            fresh += 1
        states.append({
            "timestamp": row["estimated_timestamp"],
            "observed_critics": frozenset(critics),
            "fresh_count": fresh,
            "total_count": total,
        })
    
    return states, movie_reviews["estimated_timestamp"].iloc[0]


def get_review_state_at(states, snapshot_time):
    """Binary search for the review state at snapshot_time."""
    if not states or snapshot_time < states[0]["timestamp"]:
        return set(), 0, 0
    
    lo, hi = 0, len(states) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if states[mid]["timestamp"] <= snapshot_time:
            lo = mid
        else:
            hi = mid - 1
    
    s = states[lo]
    return set(s["observed_critics"]), s["fresh_count"], s["total_count"]

In [ ]:
def backtest_movie(slug, reviews_df, movies_df, every_n_hours=1):
    """Run backtest for a single movie. Returns list of trade records."""
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]
    
    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty:
        return []
    
    market_close_time = price_df["timestamp"].iloc[-1]
    resolution = get_resolution(price_df)
    
    training_slugs = default_training_slugs(
        movies_df, exclude_slug=slug, before_date=bet_close_date
    )
    if len(training_slugs) < 5:
        return []
    
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)
    
    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    
    records = []
    cached_critics, cached_fresh, cached_total = set(), 0, 0
    cached_lambda, cached_p_fresh = None, None
    last_kept_ts = None
    
    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]
        
        if every_n_hours > 1 and last_kept_ts is not None:
            hours_since = (snapshot_time - last_kept_ts).total_seconds() / 3600
            if hours_since < every_n_hours:
                continue
        last_kept_ts = snapshot_time
        
        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        if hours_to_close <= 0:
            continue
        
        days_before_close = hours_to_close / 24
        
        observed_critics, fresh_count, total_count = get_review_state_at(
            review_states, snapshot_time
        )
        
        state_changed = (total_count != cached_total)
        
        if state_changed or cached_lambda is None:
            cached_critics = observed_critics
            cached_fresh = fresh_count
            cached_total = total_count
            
            first_review_dbc = None
            if first_review_ts is not None and total_count > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                observed_critics, observed_count=total_count,
                first_review_dbc=first_review_dbc,
            )
            cached_p_fresh = estimate_p_fresh(
                profiles, observed_critics, fresh_count, total_count,
            )
        else:
            first_review_dbc = None
            if first_review_ts is not None and cached_total > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                cached_critics, observed_count=cached_total,
                first_review_dbc=first_review_dbc,
            )
        
        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            
            if pd.isna(market_price):
                continue
            
            resolved = resolution.get(thresh)
            if resolved is None:
                continue
            
            try:
                result = compute_edge(
                    threshold=thresh,
                    market_price=market_price,
                    fresh_count=cached_fresh,
                    total_count=cached_total,
                    hours_to_close=hours_to_close,
                    lambda_rate=cached_lambda,
                    p_fresh=cached_p_fresh,
                )
            except (ValueError, Exception):
                continue
            
            # Current displayed score at snapshot time
            current_score = (cached_fresh / cached_total * 100) if cached_total > 0 else None
            
            records.append({
                "slug": slug,
                "snapshot_time": snapshot_time,
                "hours_to_close": hours_to_close,
                "threshold": thresh,
                "market_price": market_price,
                "model_p_yes": result["p_yes"],
                "edge_cents": result["edge_cents"],
                "resolved_yes": resolved,
                "lambda_rate": cached_lambda,
                "p_fresh": cached_p_fresh,
                "fresh_count": cached_fresh,
                "total_count": cached_total,
                "expected_reviews": result["expected_reviews"],
                "current_score": current_score,
            })
    
    return records

In [ ]:
import time

EVERY_N_HOURS = 24  # daily snapshots

all_records = []
slugs = movies_bt["Slug"].tolist()
skipped = []
t0 = time.time()

for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)", end="", flush=True)
    try:
        records = backtest_movie(slug, reviews_df, movies_df, every_n_hours=EVERY_N_HOURS)
        all_records.extend(records)
    except Exception as e:
        skipped.append((slug, str(e)))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.0f}s. {len(all_records):,} trade evaluations across {len(slugs) - len(skipped)} movies.")
if skipped:
    print(f"Skipped {len(skipped)} movies:")
    for s, err in skipped:
        print(f"  {s}: {err}")

In [ ]:
# Build trades DataFrame with derived columns
trades = pd.DataFrame(all_records)

trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")
trades["abs_edge"] = trades["edge_cents"].abs()

# P&L per contract (cents)
trades["pnl"] = np.where(
    trades["direction"] == "Yes",
    np.where(trades["resolved_yes"], 100 - trades["market_price"], -trades["market_price"]),
    np.where(trades["resolved_yes"], -(100 - trades["market_price"]), trades["market_price"]),
)

# Score margin: how far above/below threshold the current score sits
trades["score_margin"] = trades["current_score"] - trades["threshold"]

print(f"Shape: {trades.shape}")
print(f"Movies: {trades['slug'].nunique()}")
print(f"Thresholds: {sorted(trades['threshold'].unique())}")
print(f"Direction split: {trades['direction'].value_counts().to_dict()}")

## 1. No-side P&L by threshold

Core test: does No edge concentrate at higher thresholds?

In [ ]:
MIN_EDGE = 10  # cents
ACTION_WINDOW = (24, 120)  # T-5d to T-1d in hours

# No-side trades in the action window with sufficient edge
no_trades = trades[
    (trades["direction"] == "No") &
    (trades["abs_edge"] >= MIN_EDGE) &
    (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
    (trades["hours_to_close"] <= ACTION_WINDOW[1])
].copy()

print(f"No-side trades (min_edge={MIN_EDGE}c, T-5d to T-1d): {len(no_trades)}")
print()

# P&L by individual threshold
by_thresh = no_trades.groupby("threshold").agg(
    trades=("pnl", "size"),
    win_rate=("pnl", lambda x: (x > 0).mean()),
    total_pnl=("pnl", "sum"),
    mean_pnl=("pnl", "mean"),
    mean_edge=("abs_edge", "mean"),
    movies=("slug", "nunique"),
).reset_index()

print("=== No-side P&L by threshold ===")
print(by_thresh.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

In [ ]:
# Visualize: win rate + mean P&L by threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(by_thresh["threshold"], by_thresh["win_rate"] * 100, width=4, color="steelblue")
ax1.axhline(50, color="k", linewidth=0.5, linestyle="--")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Win Rate (%)")
ax1.set_title(f"No-side Win Rate by Threshold (min_edge={MIN_EDGE}c)")
for _, r in by_thresh.iterrows():
    ax1.text(r["threshold"], r["win_rate"] * 100 + 1, f'{r["trades"]:.0f}',
             ha="center", va="bottom", fontsize=8, color="gray")

colors = ["green" if x > 0 else "red" for x in by_thresh["mean_pnl"]]
ax2.bar(by_thresh["threshold"], by_thresh["mean_pnl"], width=4, color=colors)
ax2.axhline(0, color="k", linewidth=0.5)
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Mean P&L per trade (cents)")
ax2.set_title(f"No-side Mean P&L by Threshold (min_edge={MIN_EDGE}c)")

plt.tight_layout()
plt.show()

## 2. Threshold x Horizon heatmap

Does the edge concentrate in the "high threshold + late cycle" quadrant?

In [ ]:
# Threshold x Horizon: win rate and mean P&L
no_trades["horizon_bucket"] = pd.cut(
    no_trades["hours_to_close"],
    bins=[24, 48, 72, 96, 120],
    labels=["T-1d to T-2d", "T-2d to T-3d", "T-3d to T-4d", "T-4d to T-5d"],
)

# Threshold buckets (individual thresholds may be sparse, but let's try both)
no_trades["thresh_bucket"] = pd.cut(
    no_trades["threshold"],
    bins=[0, 60, 70, 80, 90, 100],
    labels=["<60", "60-70", "70-80", "80-90", "90+"],
    right=False,
)

# Heatmap: mean P&L
pivot_pnl = no_trades.groupby(["thresh_bucket", "horizon_bucket"], observed=True)["pnl"].mean().unstack()
pivot_wr = no_trades.groupby(["thresh_bucket", "horizon_bucket"], observed=True)["pnl"].apply(lambda x: (x > 0).mean()).unstack()
pivot_n = no_trades.groupby(["thresh_bucket", "horizon_bucket"], observed=True)["pnl"].size().unstack()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Mean P&L heatmap
vmax = max(abs(pivot_pnl.values[~np.isnan(pivot_pnl.values)].min()),
           abs(pivot_pnl.values[~np.isnan(pivot_pnl.values)].max()), 1)
im1 = ax1.imshow(pivot_pnl.values, aspect="auto", cmap="RdYlGn",
                  norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax))
ax1.set_xticks(range(len(pivot_pnl.columns)))
ax1.set_xticklabels(pivot_pnl.columns, rotation=45, ha="right")
ax1.set_yticks(range(len(pivot_pnl.index)))
ax1.set_yticklabels(pivot_pnl.index)
ax1.set_title(f"Mean P&L (cents) — No-side, min_edge={MIN_EDGE}c")
# Annotate with value and count
for i in range(len(pivot_pnl.index)):
    for j in range(len(pivot_pnl.columns)):
        val = pivot_pnl.values[i, j]
        n = pivot_n.values[i, j] if not np.isnan(pivot_n.values[i, j]) else 0
        if not np.isnan(val):
            ax1.text(j, i, f"{val:.0f}c\nn={n:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im1, ax=ax1)

# Win rate heatmap
im2 = ax2.imshow(pivot_wr.values * 100, aspect="auto", cmap="RdYlGn", vmin=30, vmax=100)
ax2.set_xticks(range(len(pivot_wr.columns)))
ax2.set_xticklabels(pivot_wr.columns, rotation=45, ha="right")
ax2.set_yticks(range(len(pivot_wr.index)))
ax2.set_yticklabels(pivot_wr.index)
ax2.set_title(f"Win Rate (%) — No-side, min_edge={MIN_EDGE}c")
for i in range(len(pivot_wr.index)):
    for j in range(len(pivot_wr.columns)):
        val = pivot_wr.values[i, j]
        n = pivot_n.values[i, j] if not np.isnan(pivot_n.values[i, j]) else 0
        if not np.isnan(val):
            ax2.text(j, i, f"{val*100:.0f}%\nn={n:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## 3. Fragility math visualization

Show the asymmetry: how much a single rotten review moves the score down vs a fresh review moves it up, as a function of current score.

In [ ]:
# Fragility ratio: p/(1-p) — how many times harder a rotten review hits vs a fresh review helps
pct = np.linspace(50, 95, 100)
ratio = (pct / 100) / (1 - pct / 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(pct, ratio, "b-", linewidth=2)
ax1.set_xlabel("Current Score (%)")
ax1.set_ylabel("Down/Up Impact Ratio")
ax1.set_title("Fragility: Rotten Review Impact / Fresh Review Impact")
ax1.axhline(1, color="k", linewidth=0.5, linestyle="--", label="Symmetric (50%)")
for p in [60, 70, 75, 80, 85, 90]:
    r = (p / 100) / (1 - p / 100)
    ax1.annotate(f"{p}%: {r:.1f}x", xy=(p, r), fontsize=9,
                 xytext=(p - 5, r + 0.5), arrowprops=dict(arrowstyle="->", color="gray"))
ax1.set_xlim(50, 95)
ax1.grid(alpha=0.3)

# Concrete example: at n=150 reviews, how much does score move per review?
n = 150
for p_val in [60, 70, 75, 80, 85, 90]:
    p = p_val / 100
    up_move = (1 - p) / (n + 1) * 100  # percentage points
    down_move = p / (n + 1) * 100
    ax2.barh(f"{p_val}%", -down_move, color="red", alpha=0.7, label="Rotten" if p_val == 60 else "")
    ax2.barh(f"{p_val}%", up_move, color="green", alpha=0.7, label="Fresh" if p_val == 60 else "")
    ax2.text(-down_move - 0.01, f"{p_val}%", f"-{down_move:.2f}pp", va="center", ha="right", fontsize=8)
    ax2.text(up_move + 0.01, f"{p_val}%", f"+{up_move:.2f}pp", va="center", ha="left", fontsize=8)

ax2.axvline(0, color="k", linewidth=0.5)
ax2.set_xlabel("Score change per review (percentage points)")
ax2.set_title(f"Impact of one review at n={n} reviews")
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Threshold filtering: cumulative P&L if we only bet No above threshold X

If the hypothesis holds, filtering to "only bet No on thresholds >= X" should improve per-trade quality. Does it?

In [ ]:
# What happens if we only bet No on thresholds >= some cutoff?
# Sweep the cutoff from 45 to 85 and show how P&L metrics change.

cutoffs = list(range(45, 90, 5))
rows = []
for cutoff in cutoffs:
    t = no_trades[no_trades["threshold"] >= cutoff]
    if len(t) < 5:
        continue
    rows.append({
        "min_threshold": cutoff,
        "trades": len(t),
        "win_rate": (t["pnl"] > 0).mean(),
        "total_pnl": t["pnl"].sum(),
        "mean_pnl": t["pnl"].mean(),
        "movies": t["slug"].nunique(),
    })

cutoff_df = pd.DataFrame(rows)
print("=== No-side P&L with minimum threshold filter ===")
print(cutoff_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

# Plot
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

ax1.plot(cutoff_df["min_threshold"], cutoff_df["win_rate"] * 100, "o-", color="steelblue")
ax1.set_xlabel("Minimum Threshold")
ax1.set_ylabel("Win Rate (%)")
ax1.set_title("Win Rate vs Minimum Threshold Filter")
ax1.grid(alpha=0.3)

ax2.plot(cutoff_df["min_threshold"], cutoff_df["mean_pnl"], "o-", color="green")
ax2.set_xlabel("Minimum Threshold")
ax2.set_ylabel("Mean P&L per trade (cents)")
ax2.set_title("Per-Trade Quality vs Min Threshold")
ax2.grid(alpha=0.3)

ax3.plot(cutoff_df["min_threshold"], cutoff_df["total_pnl"], "o-", color="purple")
ax3.set_xlabel("Minimum Threshold")
ax3.set_ylabel("Total P&L (cents)")
ax3.set_title("Total P&L vs Min Threshold")
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Score margin at time of bet

What does the score look like relative to the threshold when our model signals No? If the model is catching fragility, we should see many signals where the current score is *above* the threshold (market thinks "safe") but the model disagrees.

In [ ]:
# Score margin: current_score - threshold at time of No signal
# Positive margin = score is ABOVE threshold (market thinks safe, model says No anyway)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of score margins for winning vs losing No trades
winners = no_trades[no_trades["pnl"] > 0]
losers = no_trades[no_trades["pnl"] <= 0]

bins = np.arange(-30, 35, 2)
ax1.hist(winners["score_margin"], bins=bins, alpha=0.6, label=f"Winners (n={len(winners)})", color="green")
ax1.hist(losers["score_margin"], bins=bins, alpha=0.6, label=f"Losers (n={len(losers)})", color="red")
ax1.axvline(0, color="k", linewidth=1, linestyle="--", label="Score = Threshold")
ax1.set_xlabel("Score Margin (current score - threshold)")
ax1.set_ylabel("Count")
ax1.set_title("Score Margin at No Signal")
ax1.legend()

# Win rate by score margin bucket
no_trades["margin_bucket"] = pd.cut(
    no_trades["score_margin"],
    bins=[-30, -10, -5, 0, 5, 10, 20, 30],
    labels=["<-10", "-10 to -5", "-5 to 0", "0 to 5", "5 to 10", "10 to 20", "20+"],
)
margin_stats = no_trades.groupby("margin_bucket", observed=True).agg(
    trades=("pnl", "size"),
    win_rate=("pnl", lambda x: (x > 0).mean()),
    mean_pnl=("pnl", "mean"),
).reset_index()

colors = ["green" if wr > 0.5 else "red" for wr in margin_stats["win_rate"]]
bars = ax2.bar(range(len(margin_stats)), margin_stats["win_rate"] * 100, color=colors, alpha=0.7)
ax2.set_xticks(range(len(margin_stats)))
ax2.set_xticklabels(margin_stats["margin_bucket"], rotation=45, ha="right")
ax2.axhline(50, color="k", linewidth=0.5, linestyle="--")
ax2.set_xlabel("Score Margin Bucket")
ax2.set_ylabel("Win Rate (%)")
ax2.set_title("No-side Win Rate by Score Margin")
for i, (_, r) in enumerate(margin_stats.iterrows()):
    ax2.text(i, r["win_rate"] * 100 + 1, f'n={r["trades"]:.0f}', ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.show()

print("\n=== No-side stats by score margin ===")
print(margin_stats.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

## 6. Expected reviews × threshold interaction

The second part of the hypothesis: late-cycle bets where the model predicts many remaining reviews should be especially profitable at high thresholds (more reviews = more chances for rotten reviews to exploit fragility).

In [ ]:
# Expected reviews × threshold interaction
no_trades["exp_reviews_bucket"] = pd.cut(
    no_trades["expected_reviews"],
    bins=[0, 10, 25, 50, 100, 500],
    labels=["0-10", "10-25", "25-50", "50-100", "100+"],
)

# Heatmap: win rate by threshold bucket × expected reviews
pivot_wr2 = no_trades.groupby(["thresh_bucket", "exp_reviews_bucket"], observed=True)["pnl"].apply(
    lambda x: (x > 0).mean()
).unstack()
pivot_n2 = no_trades.groupby(["thresh_bucket", "exp_reviews_bucket"], observed=True)["pnl"].size().unstack()
pivot_pnl2 = no_trades.groupby(["thresh_bucket", "exp_reviews_bucket"], observed=True)["pnl"].mean().unstack()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Win rate
im1 = ax1.imshow(pivot_wr2.values * 100, aspect="auto", cmap="RdYlGn", vmin=30, vmax=100)
ax1.set_xticks(range(len(pivot_wr2.columns)))
ax1.set_xticklabels(pivot_wr2.columns)
ax1.set_yticks(range(len(pivot_wr2.index)))
ax1.set_yticklabels(pivot_wr2.index)
ax1.set_xlabel("Expected Remaining Reviews")
ax1.set_title("Win Rate: Threshold × Expected Reviews")
for i in range(len(pivot_wr2.index)):
    for j in range(len(pivot_wr2.columns)):
        val = pivot_wr2.values[i, j]
        n = pivot_n2.values[i, j] if not np.isnan(pivot_n2.values[i, j]) else 0
        if not np.isnan(val):
            ax1.text(j, i, f"{val*100:.0f}%\nn={n:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im1, ax=ax1)

# Mean P&L
vmax2 = max(abs(np.nanmin(pivot_pnl2.values)), abs(np.nanmax(pivot_pnl2.values)), 1)
im2 = ax2.imshow(pivot_pnl2.values, aspect="auto", cmap="RdYlGn",
                  norm=TwoSlopeNorm(vmin=-vmax2, vcenter=0, vmax=vmax2))
ax2.set_xticks(range(len(pivot_pnl2.columns)))
ax2.set_xticklabels(pivot_pnl2.columns)
ax2.set_yticks(range(len(pivot_pnl2.index)))
ax2.set_yticklabels(pivot_pnl2.index)
ax2.set_xlabel("Expected Remaining Reviews")
ax2.set_title("Mean P&L: Threshold × Expected Reviews")
for i in range(len(pivot_pnl2.index)):
    for j in range(len(pivot_pnl2.columns)):
        val = pivot_pnl2.values[i, j]
        n = pivot_n2.values[i, j] if not np.isnan(pivot_n2.values[i, j]) else 0
        if not np.isnan(val):
            ax2.text(j, i, f"{val:.0f}c\nn={n:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## 7. Distribution of final scores

Baseline context: where do movies actually end up? If most cluster around 70-80%, that tells us which thresholds are in the "fragile zone."

In [ ]:
# Final score distribution across all movies
# Use the last snapshot's score for each movie as proxy for final score
final_scores = trades.groupby("slug").last()[["current_score"]].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(final_scores["current_score"], bins=np.arange(0, 105, 5), color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(final_scores["current_score"].median(), color="red", linewidth=2, linestyle="--",
           label=f'Median: {final_scores["current_score"].median():.0f}%')
ax.axvline(final_scores["current_score"].mean(), color="orange", linewidth=2, linestyle="--",
           label=f'Mean: {final_scores["current_score"].mean():.0f}%')
ax.set_xlabel("Final Tomatometer Score (%)")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of Final Scores (n={len(final_scores)} movies)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final score stats:")
print(final_scores["current_score"].describe().round(1))

# Threshold Fragility Analysis

**Hypothesis:** The No-only edge concentrates at high thresholds because high percentages are mathematically fragile.

At score = f/n, a rotten review moves the score down by `p/(n+1)` while a fresh review moves it up by `(1-p)/(n+1)`. The down/up impact ratio is `p/(1-p)` — at 75% that's 3:1, at 85% it's 5.7:1, at 90% it's 9:1. Bettors who eyeball "85% is above 80%" underestimate this fragility. Our Poisson-binomial model does the exact math.

**Second factor:** Late in the betting cycle, our lambda model accounts for remaining review volume that bettors may ignore. The combination of high threshold + remaining reviews should be where No edge is strongest.

**Test:** Slice the existing backtest trades by threshold level and hours_to_close to see where No-side P&L concentrates.